# Sinkhole Susceptibility Mapping — Central Florida
## CAP 5937 Final Project | Spring 2026

**Objective:** Build and compare seven machine learning classifiers to predict sinkhole susceptibility using 13 hydrogeological and topographic conditioning factors.

**Dataset:** 5,759 samples — 2,900 confirmed sinkhole locations (positive class) and 2,859 pseudo-absence points (negative class) generated using a 500 m exclusion buffer to 5,000 m outer boundary spatial sampling protocol.

---
**Table of Contents**
1. [Environment Setup](#1-environment-setup)
2. [Data Loading & Exploration](#2-data-loading--exploration)
3. [Preprocessing & Train-Test Split](#3-preprocessing--train-test-split)
4. [Model Training & Evaluation](#4-model-training--evaluation)
5. [Results Summary](#5-results-summary)
6. [Feature Importance Analysis](#6-feature-importance-analysis)
7. [Visualizations](#7-visualizations)


## 1. Environment Setup

In [ ]:
# Install required libraries if not already present
# Uncomment the line below if running in a fresh environment
# !pip install scikit-learn matplotlib seaborn pandas numpy

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    ConfusionMatrixDisplay,
)

print("All libraries loaded successfully.")
print(f"  pandas   : {pd.__version__}")
print(f"  numpy    : {np.__version__}")
print(f"  sklearn  : {__import__('sklearn').__version__}")
print(f"  matplotlib: {matplotlib.__version__}")


## 2. Data Loading & Exploration

The CSV contains 5,759 georeferenced point samples across Central Florida with 13 conditioning factors and a binary label:
- **Label = 1** → confirmed sinkhole location (Florida Geological Survey database)
- **Label = 0** → pseudo-absence point (500 m – 5,000 m buffer from nearest sinkhole)


### Data file path

The data file is loaded from your OneDrive Desktop folder.
If you move the notebook or the file, update the `DATA_PATH` variable in the next cell.

```
C:\Users\os084369\OneDrive - University of Central Florida\Desktop\Applied Machine learning\Sinkholes data.csv
```


In [ ]:
# ── Data file path — update this if you move the file ────────────────────────
DATA_PATH = r"C:\Users\os084369\OneDrive - University of Central Florida\Desktop\Applied Machine learning\Sinkholes data.csv"

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nLabel distribution:")
print(df['Label'].value_counts().rename({1.0: 'Sinkhole (1)', 0.0: 'Non-Sinkhole (0)'}))
print(f"\nClass balance: {df['Label'].mean()*100:.1f}% positive")


In [ ]:
# ── Define feature list ──────────────────────────────────────────────────────
FEATURES = [
    'Annual_Precipitation',
    'Distance_to_Nearest_Karst_Feature',
    'Distance_to_Nearest_Water_body',
    'Elevation_(m)',
    'Hydrualic_Head_Difference',
    'IAS_Thickness',
    'LULC',
    'Overburden_Thickness',
    'SAS_Thickness',
    'Shear_Waves_Velocities',
    'Slope',
    'Surface_Geology',
    'TPI',
]

FEATURE_LABELS = [
    'Annual Precip.',
    'Dist. to Karst Feature',
    'Dist. to Water Body',
    'Elevation (m)',
    'Hydraulic Head Diff.',
    'IAS Thickness',
    'LULC',
    'Overburden Thickness',
    'SAS Thickness',
    'Shear Wave Vel.',
    'Slope',
    'Surface Geology',
    'TPI',
]

TARGET = 'Label'

print(f"Feature count : {len(FEATURES)}")
print(f"Target column : {TARGET}")


In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
df[FEATURES].describe().round(3)


In [ ]:
# ── Missing values check ─────────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "  None — dataset is complete.")


In [ ]:
# ── Pearson correlation with label ───────────────────────────────────────────
corr = (
    df[FEATURES + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .abs()
    .sort_values(ascending=False)
)

print("Absolute Pearson |r| with Label (all features, ranked):")
for feat, val in corr.items():
    bar = '█' * int(val * 40)
    print(f"  {feat:<42} {val:.4f}  {bar}")


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
corr_matrix = df[FEATURES].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 8},
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    xticklabels=FEATURE_LABELS,
    yticklabels=FEATURE_LABELS,
    linewidths=0.4, ax=ax
)
ax.set_title('Feature Correlation Matrix\n(lower triangle)', fontsize=13, fontweight='bold', pad=14)
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Preprocessing & Train-Test Split

**Scaling strategy:**
- Tree-based models (Random Forest, Gradient Boosting, Extra Trees, AdaBoost) → **no scaling** (splits are invariant to monotonic transformations)
- Distance/gradient-based models (Logistic Regression, SVM, MLP) → **StandardScaler** fit on training data only

The scaler is fit **exclusively on training data** to prevent any information from the test set leaking into the normalization parameters.


In [ ]:
# ── Train / test split (80 / 20, stratified) ─────────────────────────────────
X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # preserve label ratio in both splits
)

print(f"Training set : {X_train.shape[0]} samples ({y_train.mean()*100:.1f}% positive)")
print(f"Test set     : {X_test.shape[0]} samples ({y_test.mean()*100:.1f}% positive)")


In [ ]:
# ── StandardScaler (fit on train only) ───────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaler fitted on training data.")
print(f"  Feature means  (first 3): {scaler.mean_[:3].round(3)}")
print(f"  Feature stdevs (first 3): {scaler.scale_[:3].round(3)}")


## 4. Model Training & Evaluation

Seven classifiers are trained and evaluated across five metrics:

| Metric | Description |
|---|---|
| **Accuracy** | Overall % of correctly classified samples |
| **AUC-ROC** | Area under ROC curve — threshold-independent discrimination *(primary metric)* |
| **F1-Score** | Harmonic mean of precision and recall |
| **Sensitivity** | True Positive Rate — % of actual sinkholes correctly identified |
| **Specificity** | True Negative Rate — % of stable locations correctly identified |

AUC-ROC is used as the primary ranking metric because susceptibility mapping applications operate at different decision thresholds depending on risk tolerance.


In [ ]:
# ── Model registry ────────────────────────────────────────────────────────────
# Each entry: (model_object, needs_scaling)
MODEL_REGISTRY = {
    'Logistic Regression': (
        LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42),
        True
    ),
    'Random Forest': (
        RandomForestClassifier(
            n_estimators=300, max_depth=15, min_samples_leaf=2,
            criterion='gini', random_state=42, n_jobs=-1
        ),
        False
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=5,
            subsample=0.8, random_state=42
        ),
        False
    ),
    'Extra Trees': (
        ExtraTreesClassifier(
            n_estimators=300, max_depth=15,
            random_state=42, n_jobs=-1
        ),
        False
    ),
    'AdaBoost': (
        AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42),
        False
    ),
    'SVM (RBF)': (
        SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42),
        True
    ),
    'MLP Neural Net': (
        MLPClassifier(
            hidden_layer_sizes=(128, 64, 32), alpha=0.001,
            max_iter=500, early_stopping=True, random_state=42
        ),
        True
    ),
}

print(f"Models registered: {len(MODEL_REGISTRY)}")
for name in MODEL_REGISTRY:
    _, scaled = MODEL_REGISTRY[name]
    print(f"  {'[scaled]' if scaled else '[raw]  ':10s} {name}")


In [ ]:
# ── Train all models and compute metrics ─────────────────────────────────────
results   = {}   # metrics per model
probs     = {}   # predicted probabilities for ROC curves
preds     = {}   # predicted labels for confusion matrices

print(f"{'Model':<22} {'Acc%':>7} {'AUC':>7} {'F1%':>7} {'Sens%':>7} {'Spec%':>7}")
print("-" * 60)

for name, (model, scaled) in MODEL_REGISTRY.items():
    Xtr = X_train_scaled if scaled else X_train
    Xte = X_test_scaled  if scaled else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    acc  = accuracy_score(y_test, y_pred) * 100
    auc  = roc_auc_score(y_test, y_prob)
    f1   = f1_score(y_test, y_pred) * 100
    sens = tp / (tp + fn) * 100
    spec = tn / (tn + fp) * 100

    results[name] = {
        'Accuracy':    round(acc,  2),
        'AUC':         round(auc,  4),
        'F1':          round(f1,   2),
        'Sensitivity': round(sens, 2),
        'Specificity': round(spec, 2),
    }
    probs[name] = y_prob
    preds[name] = y_pred

    print(f"{name:<22} {acc:>7.2f} {auc:>7.4f} {f1:>7.2f} {sens:>7.2f} {spec:>7.2f}")


## 5. Results Summary

In [ ]:
# ── Results as a styled DataFrame ────────────────────────────────────────────
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('AUC', ascending=False)

# Highlight best value per column
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: #d4edda; font-weight: bold' if v else '' for v in is_max]

styled = (
    results_df
    .style
    .apply(highlight_max)
    .format({
        'Accuracy':    '{:.2f}%',
        'AUC':         '{:.4f}',
        'F1':          '{:.2f}%',
        'Sensitivity': '{:.2f}%',
        'Specificity': '{:.2f}%',
    })
    .set_caption('Model Performance — Held-out Test Set (n = 1,152) | Green = best per column')
)
styled


In [ ]:
# ── Best model summary ────────────────────────────────────────────────────────
best_name = max(results, key=lambda n: results[n]['AUC'])
best      = results[best_name]

print(f"Best model by AUC: {best_name}")
print(f"  Accuracy    : {best['Accuracy']:.2f}%")
print(f"  AUC-ROC     : {best['AUC']:.4f}")
print(f"  F1-Score    : {best['F1']:.2f}%")
print(f"  Sensitivity : {best['Sensitivity']:.2f}%  (True Positive Rate)")
print(f"  Specificity : {best['Specificity']:.2f}%  (True Negative Rate)")
print()
print(f"  Interpretation: of every 1,000 actual sinkholes,")
print(f"  the model correctly flags {best['Sensitivity']*10:.0f} while missing {(100-best['Sensitivity'])*10:.0f}.")


## 6. Feature Importance Analysis

Feature importances are computed using **Mean Decrease in Impurity (MDI)** from the fitted Random Forest model.
Results are cross-validated against Gradient Boosting importances — both methods agree on the top-ranked factors.


In [ ]:
# ── Random Forest feature importance ─────────────────────────────────────────
rf_model = MODEL_REGISTRY['Random Forest'][0]
fi_rf    = rf_model.feature_importances_

fi_df = (
    pd.DataFrame({'Feature': FEATURES, 'Label': FEATURE_LABELS, 'Importance': fi_rf})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

print("Random Forest — Feature Importances (MDI), ranked:")
print(f"{'Rank':<5} {'Feature':<42} {'Importance':>10}")
print("-" * 60)
for i, row in fi_df.iterrows():
    bar = '█' * int(row['Importance'] * 100)
    print(f"  {i+1:<4} {row['Feature']:<42} {row['Importance']:.4f}  {bar}")


In [ ]:
# ── Gradient Boosting feature importance (cross-check) ───────────────────────
gb_model = MODEL_REGISTRY['Gradient Boosting'][0]
fi_gb    = gb_model.feature_importances_

fi_gb_df = (
    pd.DataFrame({'Feature': FEATURES, 'Importance_GB': fi_gb})
    .sort_values('Importance_GB', ascending=False)
    .reset_index(drop=True)
)

# Rank agreement
rf_ranks = {f: i+1 for i, f in enumerate(fi_df['Feature'])}
gb_ranks = {r['Feature']: i+1 for i, r in fi_gb_df.iterrows()}

print("Rank comparison — RF vs Gradient Boosting (top 6):")
print(f"{'Feature':<42} {'RF rank':>8} {'GB rank':>8}")
print("-" * 60)
for feat in fi_df['Feature'][:6]:
    print(f"  {feat:<42} {rf_ranks[feat]:>6}     {gb_ranks[feat]:>6}")


## 7. Visualizations

Five figures are generated and saved as PNG files:
1. ROC curves — all models
2. Performance bar chart — Accuracy, AUC, F1
3. Feature importance — Random Forest (horizontal bar)
4. Confusion matrix — best model (Random Forest)
5. Sensitivity vs. Specificity — grouped bar chart


In [ ]:
# ── Shared plot settings ─────────────────────────────────────────────────────
COLORS = [
    '#2196F3',  # Logistic Regression  — blue
    '#4CAF50',  # Random Forest        — green
    '#FF5722',  # Gradient Boosting    — deep orange
    '#9C27B0',  # Extra Trees          — purple
    '#FF9800',  # AdaBoost             — amber
    '#00BCD4',  # SVM (RBF)            — cyan
    '#E91E63',  # MLP Neural Net       — pink
]

MODEL_NAMES = list(results.keys())

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
})
print("Plot settings applied.")


In [ ]:
# ── Figure 1: ROC curves ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for name, color in zip(MODEL_NAMES, COLORS):
    fpr, tpr, _ = roc_curve(y_test, probs[name])
    auc_val = results[name]['AUC']
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name}  (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random chance (AUC = 0.50)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Sinkhole Susceptibility, Central Florida', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('fig_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_roc.png")


In [ ]:
# ── Figure 2: Performance bar chart (Accuracy, AUC, F1) ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics    = ['Accuracy', 'AUC', 'F1']
ylabels    = ['Accuracy (%)', 'AUC', 'F1-Score (%)']

for i, (metric, ylabel) in enumerate(zip(metrics, ylabels)):
    vals = [results[n][metric] for n in MODEL_NAMES]
    bars = axes[i].bar(MODEL_NAMES, vals, color=COLORS, edgecolor='white', linewidth=0.7)

    axes[i].set_title(metric, fontweight='bold', fontsize=12)
    axes[i].set_ylabel(ylabel, fontsize=10)
    axes[i].set_ylim(0, max(vals) * 1.15)
    axes[i].tick_params(axis='x', rotation=35, labelsize=8)

    for bar, val in zip(bars, vals):
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f'{val}', ha='center', va='bottom', fontsize=8, fontweight='bold'
        )

fig.suptitle('Model Performance Comparison — Sinkhole Susceptibility Mapping',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_performance.png")


In [ ]:
# ── Figure 3: RF feature importance ──────────────────────────────────────────
fi_sorted = fi_df.sort_values('Importance')   # ascending for horizontal bar

cmap_vals  = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(fi_sorted)))

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(fi_sorted['Label'], fi_sorted['Importance'], color=cmap_vals, edgecolor='white')

for bar, val in zip(bars, fi_sorted['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Mean Decrease in Impurity (Feature Importance)', fontsize=11)
ax.set_title('Random Forest Feature Importance\nSinkhole Susceptibility Conditioning Factors',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, fi_sorted['Importance'].max() * 1.18)

plt.tight_layout()
plt.savefig('fig_fi_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_fi_rf.png")


In [ ]:
# ── Figure 4: Confusion matrix — best model (Random Forest) ──────────────────
cm = confusion_matrix(y_test, preds['Random Forest'])

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Non-Sinkhole\n(Pseudo)', 'Sinkhole']
)
disp.plot(ax=ax, cmap='Blues', colorbar=True)

ax.set_title('Confusion Matrix — Random Forest (Best Model)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)

plt.tight_layout()
plt.savefig('fig_cm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_cm.png")

tn, fp, fn, tp = cm.ravel()
print(f"\n  True Positives  (sinkhole correctly flagged) : {tp}")
print(f"  False Negatives (sinkhole missed)            : {fn}")
print(f"  True Negatives  (stable correctly cleared)   : {tn}")
print(f"  False Positives (false alarm)                : {fp}")


In [ ]:
# ── Figure 5: Sensitivity vs. Specificity ────────────────────────────────────
sens_vals = [results[n]['Sensitivity'] for n in MODEL_NAMES]
spec_vals = [results[n]['Specificity'] for n in MODEL_NAMES]

x = np.arange(len(MODEL_NAMES))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, sens_vals, w, label='Sensitivity (TPR)', color='#4CAF50', edgecolor='white')
b2 = ax.bar(x + w/2, spec_vals, w, label='Specificity (TNR)', color='#2196F3', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(MODEL_NAMES, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Sensitivity vs. Specificity per Model', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 100)

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('fig_sens_spec.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_sens_spec.png")


## Summary of Outputs

| File | Description |
|---|---|
| `fig_correlation_heatmap.png` | Pearson correlation matrix across all 13 features |
| `fig_roc.png` | ROC curves for all seven models |
| `fig_performance.png` | Accuracy / AUC / F1 bar chart comparison |
| `fig_fi_rf.png` | Random Forest feature importances (MDI) |
| `fig_cm.png` | Confusion matrix — Random Forest |
| `fig_sens_spec.png` | Sensitivity vs. Specificity per model |

---
**Best model: Random Forest**
- AUC = 0.8907 | Accuracy = 80.56% | F1 = 81.14%
- Sensitivity = 83.10% | Specificity = 77.97%

**Top conditioning factor: LULC (19.1% importance)**
— followed by Distance to Karst Feature (8.9%) and Overburden Thickness (8.6%)
